In [1]:
# (필요 시) 설치
# !pip -q install sqlalchemy psycopg2-binary pandas

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from typing import Dict, Tuple, List, Optional

import pandas as pd
from sqlalchemy import create_engine, text

KST = ZoneInfo("Asia/Seoul")

# =========================
# 고정 실행 조건
# =========================
PROD_DAY = "20260130"  # 고정
STATIONS = ["Vision1", "Vision2"]
GOODORBAD_VALUE = "GoodFile"

# =========================
# DB 설정 (사양서)
# =========================
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TESTLOG_SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
TESTLOG_TABLE  = "fct_vision_testlog_txt_processing_history"

REMARK_SCHEMA = "g_production_film"
REMARK_TABLE  = "remark_info"

def make_engine():
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(url)

In [14]:
@dataclass(frozen=True)
class ShiftWindow:
    prod_day: str
    shift_type: str   # "day" or "night"
    start_dt: datetime
    end_dt: datetime  # inclusive

def shift_window(prod_day: str, shift_type: str) -> ShiftWindow:
    d = datetime.strptime(prod_day, "%Y%m%d").date()
    if shift_type == "day":
        start_dt = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
        end_dt   = datetime(d.year, d.month, d.day, 20, 29, 59, tzinfo=KST)
    elif shift_type == "night":
        start_dt = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
        d2 = (datetime(d.year, d.month, d.day) + timedelta(days=1)).date()
        end_dt   = datetime(d2.year, d2.month, d2.day, 8, 29, 59, tzinfo=KST)
    else:
        raise ValueError("shift_type must be 'day' or 'night'")
    return ShiftWindow(prod_day=prod_day, shift_type=shift_type, start_dt=start_dt, end_dt=end_dt)

# 시간대(2시간 단위, inclusive)
DAY_BANDS = [
    ("A시간대(08:30:00 ~ 10:29:59)", 0,  2*3600 - 1),
    ("B시간대(10:30:00 ~ 12:29:59)", 2*3600,  4*3600 - 1),
    ("C시간대(12:30:00 ~ 14:29:59)", 4*3600,  6*3600 - 1),
    ("D시간대(14:30:00 ~ 16:29:59)", 6*3600,  8*3600 - 1),
    ("E시간대(16:30:00 ~ 18:29:59)", 8*3600, 10*3600 - 1),
    ("F시간대(18:30:00 ~ 20:29:59)",10*3600, 12*3600 - 1),
]

NIGHT_BANDS = [
    ("A'시간대(20:30:00 ~ 22:29:59)", 0,  2*3600 - 1),
    ("B'시간대(22:30:00 ~ 00:29:59)", 2*3600,  4*3600 - 1),
    ("C'시간대(00:30:00 ~ 02:29:59)", 4*3600,  6*3600 - 1),
    ("D'시간대(02:30:00 ~ 04:29:59)", 6*3600,  8*3600 - 1),
    ("E'시간대(04:30:00 ~ 06:29:59)", 8*3600, 10*3600 - 1),
    ("F'시간대(06:30:00 ~ 08:29:59)",10*3600, 12*3600 - 1),
]

def assign_band(end_ts: pd.Series, window: ShiftWindow) -> pd.Series:
    """
    end_ts: tz-aware KST pandas datetime
    return: band label (Korean string) or NA
    """
    delta_sec = (end_ts - window.start_dt).dt.total_seconds()
    bands = DAY_BANDS if window.shift_type == "day" else NIGHT_BANDS

    out = pd.Series(pd.NA, index=end_ts.index, dtype="object")
    for label, lo, hi in bands:
        mask = (delta_sec >= lo) & (delta_sec <= hi)
        out.loc[mask] = label
    return out

In [15]:
def normalize_time_str(x: object) -> Optional[str]:
    """
    end_time이 'HH:MI:SS' 또는 'HH:MI:SS.xxx' 또는 time 타입일 수 있으니
    최종적으로 'HH:MI:SS' 문자열로 정규화.
    """
    if x is None or (isinstance(x, float) and pd.isna(x)) or (isinstance(x, str) and x.strip() == ""):
        return None
    s = str(x).strip()
    if "." in s:
        s = s.split(".", 1)[0]  # ms 제거
    # time 문자열이 'HH:MI:SS'가 아니면 여기서 추가 보정 가능
    return s

def build_end_ts_kst(df: pd.DataFrame, end_day_col: str, end_time_col: str) -> pd.Series:
    t = df[end_time_col].map(normalize_time_str)
    dt_str = df[end_day_col].astype(str).str.strip() + " " + t.astype(str)
    end_ts = pd.to_datetime(dt_str, format="%Y%m%d %H:%M:%S", errors="coerce").dt.tz_localize(KST)
    return end_ts

def pick_first_existing(cols: List[str], candidates: List[str]) -> str:
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise KeyError(f"Cannot find any of candidates: {candidates}. Available={cols}")

def normalize_passfail(s: pd.Series) -> pd.Series:
    """
    PASS/FAIL 값이 다양한 경우를 흡수.
    - PASS: 'PASS', 'P', 'OK', 'TRUE', '1' 등
    - FAIL: 'FAIL', 'F', 'NG', 'FALSE', '0' 등
    """
    x = s.astype(str).str.strip().str.upper()

    is_pass = x.isin(["PASS", "P", "OK", "TRUE", "1"]) | x.str.contains("PASS", na=False)
    is_fail = x.isin(["FAIL", "F", "NG", "FALSE", "0"]) | x.str.contains("FAIL", na=False)

    out = pd.Series(pd.NA, index=s.index, dtype="object")
    out.loc[is_pass] = "PASS"
    out.loc[is_fail] = "FAIL"
    return out

def passfail_cell(pass_cnt: int, fail_cnt: int) -> str:
    return f"PASS: {int(pass_cnt)}, FAIL: {int(fail_cnt)}"

In [16]:
def load_remark_map(engine) -> Dict[str, Tuple[str, str]]:
    q = text(f"""
        SELECT barcode_information, pn, remark
        FROM {REMARK_SCHEMA}.{REMARK_TABLE}
    """)
    m = pd.read_sql(q, engine)

    # key는 barcode_information 컬럼의 속성값(예: 'J')
    m["barcode_information"] = m["barcode_information"].astype(str).str.strip()
    m["pn"] = m["pn"].astype(str).str.strip()
    m["remark"] = m["remark"].astype(str).str.strip()

    mp: Dict[str, Tuple[str, str]] = {}
    for _, r in m.iterrows():
        k = r["barcode_information"]
        if k and k != "nan":
            mp[k] = (r["pn"], r["remark"])
    return mp

In [17]:
def load_shift_df(engine, window: ShiftWindow) -> pd.DataFrame:
    # end_day 범위를 줄여서 가져오기 (day: 20260130만 / night: 20260130~20260131)
    end_days = [window.start_dt.strftime("%Y%m%d"), window.end_dt.strftime("%Y%m%d")]
    end_days = sorted(set(end_days))

    q = text(f"""
        SELECT *
        FROM {TESTLOG_SCHEMA}.{TESTLOG_TABLE}
        WHERE station = ANY(:stations)
          AND goodorbad = :goodorbad
          AND end_day = ANY(:end_days)
    """)
    df = pd.read_sql(q, engine, params={"stations": STATIONS, "goodorbad": GOODORBAD_VALUE, "end_days": end_days})

    # 컬럼명 자동 감지
    barcode_col = pick_first_existing(df.columns.tolist(), ["barcode_information"])
    end_day_col = pick_first_existing(df.columns.tolist(), ["end_day"])
    end_time_col = pick_first_existing(df.columns.tolist(), ["end_time"])
    station_col = pick_first_existing(df.columns.tolist(), ["station"])
    # PASS/FAIL 결과 컬럼(환경에 따라 다를 수 있어 후보로 처리)
    result_col = pick_first_existing(df.columns.tolist(), ["result", "passfail", "pass_fail", "test_result", "pf", "judge", "status"])

    # end_ts 생성(KST)
    df["end_ts"] = build_end_ts_kst(df, end_day_col, end_time_col)
    df = df.dropna(subset=["end_ts"])

    # (3) window 필터 (inclusive)
    df = df[(df["end_ts"] >= window.start_dt) & (df["end_ts"] <= window.end_dt)].copy()

    # 결과값 PASS/FAIL 정규화
    df["passfail_norm"] = normalize_passfail(df[result_col])
    df = df.dropna(subset=["passfail_norm"])

    # (4) dedup: barcode_information 전체 문자열 기준, end_ts 최대 1건
    df[barcode_col] = df[barcode_col].astype(str).str.strip()
    df = df[df[barcode_col].notna() & (df[barcode_col] != "")].copy()

    df = df.sort_values("end_ts")
    df = df.groupby(barcode_col, as_index=False).tail(1).copy()

    # 밴드 부여
    df["time_band"] = assign_band(df["end_ts"], window)
    df = df.dropna(subset=["time_band"])

    # 필요한 컬럼만 정리
    out = df[[station_col, barcode_col, "end_ts", "time_band", "passfail_norm"]].copy()
    out = out.rename(columns={station_col: "station", barcode_col: "barcode_information"})
    out["prod_day"] = window.prod_day
    out["shift_type"] = window.shift_type
    return out

In [18]:
def apply_pn_remark(df: pd.DataFrame, remark_map: Dict[str, Tuple[str, str]]) -> pd.DataFrame:
    # barcode_information의 18번째 문자(1-based) => index 17(0-based)
    def key18(s: str) -> Optional[str]:
        if not isinstance(s, str):
            return None
        s = s.strip()
        if len(s) < 18:
            return None
        return s[17]  # 18th char

    keys = df["barcode_information"].map(key18)
    mapped = keys.map(lambda k: remark_map.get(k) if k is not None else None)

    df2 = df.copy()
    df2["pn"] = mapped.map(lambda x: x[0] if x else None)
    df2["remark"] = mapped.map(lambda x: x[1] if x else None)

    # (B) 미매칭은 제외
    df2 = df2.dropna(subset=["pn", "remark"])
    return df2

def summarize(df: pd.DataFrame, window: ShiftWindow, by_station: bool, updated_at: datetime) -> pd.DataFrame:
    """
    결과 컬럼 구성은 첨부표 형태로:
    - day overall: prod_day, shift_type, pn, remark, A~F, 합계, updated_at
    - day station : prod_day, shift_type, station, pn, remark, A~F, 합계, updated_at
    - night overall/station: A'~F'
    """
    bands = DAY_BANDS if window.shift_type == "day" else NIGHT_BANDS
    band_cols = [b[0] for b in bands]

    group_cols = ["prod_day", "shift_type"]
    if by_station:
        group_cols.append("station")
    group_cols += ["pn", "remark"]

    # band+PASS/FAIL 카운트
    g = (
        df.groupby(group_cols + ["time_band", "passfail_norm"])
          .size()
          .rename("cnt")
          .reset_index()
    )

    # PASS/FAIL pivot
    pivot = (
        g.pivot_table(index=group_cols + ["time_band"], columns="passfail_norm", values="cnt", aggfunc="sum", fill_value=0)
         .reset_index()
    )
    if "PASS" not in pivot.columns:
        pivot["PASS"] = 0
    if "FAIL" not in pivot.columns:
        pivot["FAIL"] = 0

    pivot["cell"] = pivot.apply(lambda r: passfail_cell(r["PASS"], r["FAIL"]), axis=1)

    # band별 cell pivot
    wide = (
        pivot.pivot_table(index=group_cols, columns="time_band", values="cell", aggfunc="first")
             .reset_index()
    )

    # 누락 band는 "PASS: 0, FAIL: 0"으로 채움
    for c in band_cols:
        if c not in wide.columns:
            wide[c] = passfail_cell(0, 0)
    wide = wide[group_cols + band_cols].copy()

    # 합계 계산(문자열이므로 원본 df에서 합계)
    tot = (
        df.groupby(group_cols + ["passfail_norm"])
          .size()
          .rename("cnt")
          .reset_index()
          .pivot_table(index=group_cols, columns="passfail_norm", values="cnt", aggfunc="sum", fill_value=0)
          .reset_index()
    )
    if "PASS" not in tot.columns:
        tot["PASS"] = 0
    if "FAIL" not in tot.columns:
        tot["FAIL"] = 0
    tot["합계"] = tot.apply(lambda r: passfail_cell(r["PASS"], r["FAIL"]), axis=1)
    tot = tot[group_cols + ["합계"]]

    out = wide.merge(tot, on=group_cols, how="left")

    # updated_at
    out["updated_at"] = updated_at

    # 컬럼 순서(요구대로)
    if by_station:
        ordered = ["prod_day", "shift_type", "station", "pn", "remark"] + band_cols + ["합계", "updated_at"]
    else:
        ordered = ["prod_day", "shift_type", "pn", "remark"] + band_cols + ["합계", "updated_at"]

    # 컬럼 순서(요구대로)
    out = out[ordered].copy()

    # station 정렬 우선순위 고정: Vision1 -> Vision2
    if by_station:
        out["station"] = pd.Categorical(out["station"], categories=["Vision1", "Vision2"], ordered=True)

    # ✅ 정렬 기준 변경: pn 우선 → (station) → remark
    if by_station:
        sort_cols = ["prod_day", "shift_type", "pn", "station", "remark"]
    else:
        sort_cols = ["prod_day", "shift_type", "pn", "remark"]

    out = out.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
    return out

In [19]:
engine = make_engine()

# updated_at: "DF가 만들어지고 업데이트되는 시점" -> 이번 실행 시점 KST now (4개 DF 공통)
updated_at = datetime.now(KST)

remark_map = load_remark_map(engine)

# day/night 윈도우
win_day = shift_window(PROD_DAY, "day")
win_night = shift_window(PROD_DAY, "night")

# shift별 로드 → pn/remark 적용
raw_day = load_shift_df(engine, win_day)
raw_day = apply_pn_remark(raw_day, remark_map)

raw_night = load_shift_df(engine, win_night)
raw_night = apply_pn_remark(raw_night, remark_map)

# 4개 결과 DF
df_day_overall   = summarize(raw_day,   win_day,   by_station=False, updated_at=updated_at)
df_day_station   = summarize(raw_day,   win_day,   by_station=True,  updated_at=updated_at)
df_night_overall = summarize(raw_night, win_night, by_station=False, updated_at=updated_at)
df_night_station = summarize(raw_night, win_night, by_station=True,  updated_at=updated_at)

# 출력
print("=== DAY / OVERALL ===")
display(df_day_overall)

print("=== DAY / BY STATION ===")
display(df_day_station)

print("=== NIGHT / OVERALL ===")
display(df_night_overall)

print("=== NIGHT / BY STATION ===")
display(df_night_station)

=== DAY / OVERALL ===


,prod_day,shift_type,pn,remark,A시간대(08:30:00 ~ 10:29:59),B시간대(10:30:00 ~ 12:29:59),C시간대(12:30:00 ~ 14:29:59),D시간대(14:30:00 ~ 16:29:59),E시간대(16:30:00 ~ 18:29:59),F시간대(18:30:00 ~ 20:29:59),합계,updated_at
0,20260130,day,35930927,PD,"PASS: 680, FAIL: 0","PASS: 671, FAIL: 0","PASS: 668, FAIL: 0","PASS: 524, FAIL: 0","PASS: 504, FAIL: 0","PASS: 229, FAIL: 0","PASS: 3276, FAIL: 0",2026-01-31 10:18:53.560831+09:00
1,20260130,day,35930928,PD,NaN,NaN,NaN,"PASS: 55, FAIL: 0","PASS: 153, FAIL: 0","PASS: 205, FAIL: 0","PASS: 413, FAIL: 0",2026-01-31 10:18:53.560831+09:00


=== DAY / BY STATION ===


,prod_day,shift_type,station,pn,remark,A시간대(08:30:00 ~ 10:29:59),B시간대(10:30:00 ~ 12:29:59),C시간대(12:30:00 ~ 14:29:59),D시간대(14:30:00 ~ 16:29:59),E시간대(16:30:00 ~ 18:29:59),F시간대(18:30:00 ~ 20:29:59),합계,updated_at
0,20260130,day,Vision1,35930927,PD,"PASS: 330, FAIL: 0","PASS: 317, FAIL: 0","PASS: 307, FAIL: 0","PASS: 246, FAIL: 0","PASS: 245, FAIL: 0","PASS: 95, FAIL: 0","PASS: 1540, FAIL: 0",2026-01-31 10:18:53.560831+09:00
1,20260130,day,Vision2,35930927,PD,"PASS: 350, FAIL: 0","PASS: 354, FAIL: 0","PASS: 361, FAIL: 0","PASS: 278, FAIL: 0","PASS: 259, FAIL: 0","PASS: 134, FAIL: 0","PASS: 1736, FAIL: 0",2026-01-31 10:18:53.560831+09:00
2,20260130,day,Vision1,35930928,PD,NaN,NaN,NaN,"PASS: 28, FAIL: 0","PASS: 71, FAIL: 0","PASS: 97, FAIL: 0","PASS: 196, FAIL: 0",2026-01-31 10:18:53.560831+09:00
3,20260130,day,Vision2,35930928,PD,NaN,NaN,NaN,"PASS: 27, FAIL: 0","PASS: 82, FAIL: 0","PASS: 108, FAIL: 0","PASS: 217, FAIL: 0",2026-01-31 10:18:53.560831+09:00


=== NIGHT / OVERALL ===


,prod_day,shift_type,pn,remark,A'시간대(20:30:00 ~ 22:29:59),B'시간대(22:30:00 ~ 00:29:59),C'시간대(00:30:00 ~ 02:29:59),D'시간대(02:30:00 ~ 04:29:59),E'시간대(04:30:00 ~ 06:29:59),F'시간대(06:30:00 ~ 08:29:59),합계,updated_at
0,20260130,night,35930928,PD,"PASS: 650, FAIL: 0","PASS: 591, FAIL: 0","PASS: 628, FAIL: 0","PASS: 647, FAIL: 0","PASS: 644, FAIL: 0","PASS: 466, FAIL: 0","PASS: 3626, FAIL: 0",2026-01-31 10:18:53.560831+09:00


=== NIGHT / BY STATION ===


,prod_day,shift_type,station,pn,remark,A'시간대(20:30:00 ~ 22:29:59),B'시간대(22:30:00 ~ 00:29:59),C'시간대(00:30:00 ~ 02:29:59),D'시간대(02:30:00 ~ 04:29:59),E'시간대(04:30:00 ~ 06:29:59),F'시간대(06:30:00 ~ 08:29:59),합계,updated_at
0,20260130,night,Vision1,35930928,PD,"PASS: 328, FAIL: 0","PASS: 319, FAIL: 0","PASS: 309, FAIL: 0","PASS: 326, FAIL: 0","PASS: 326, FAIL: 0","PASS: 225, FAIL: 0","PASS: 1833, FAIL: 0",2026-01-31 10:18:53.560831+09:00
1,20260130,night,Vision2,35930928,PD,"PASS: 322, FAIL: 0","PASS: 272, FAIL: 0","PASS: 319, FAIL: 0","PASS: 321, FAIL: 0","PASS: 318, FAIL: 0","PASS: 241, FAIL: 0","PASS: 1793, FAIL: 0",2026-01-31 10:18:53.560831+09:00


In [21]:
from sqlalchemy import text
import math
import pandas as pd

SAVE_SCHEMA = "i_daily_report"

T_DAY_OVERALL   = "a_day_daily_final_amount"
T_DAY_STATION   = "a_station_day_daily_final_amount"
T_NIGHT_OVERALL = "a_night_daily_final_amount"
T_NIGHT_STATION = "a_station_night_daily_final_amount"

def ensure_schema(engine, schema: str):
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema};"))

def ensure_table(engine, schema: str, table: str, columns: list[str], key_cols: list[str]):
    """
    columns: DF 컬럼명 리스트 (DB에 TEXT로 생성, updated_at만 timestamptz)
    key_cols: UNIQUE key 컬럼 리스트
    """
    ddl_cols = []
    for c in columns:
        if c == "updated_at":
            ddl_cols.append(f'"{c}" timestamptz')
        else:
            ddl_cols.append(f'"{c}" text')

    ddl_cols_sql = ",\n  ".join(ddl_cols)
    key_sql = ", ".join([f'"{c}"' for c in key_cols])

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {schema}.{table} (
      {ddl_cols_sql},
      UNIQUE ({key_sql})
    );
    """

    with engine.begin() as conn:
        conn.execute(text(create_sql))

def _to_py(v):
    """NaN -> None, pandas Timestamp -> python datetime"""
    if v is None:
        return None
    try:
        if isinstance(v, float) and math.isnan(v):
            return None
    except Exception:
        pass
    if pd.isna(v):
        return None
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    return v

def upsert_df(engine, schema: str, table: str, df: pd.DataFrame, key_cols: list[str]):
    """
    ✅ 컬럼명에 괄호/콜론/따옴표/한글 등이 있어도 안전하게 UPSERT.
    - 컬럼명은 "..."로 quoting
    - 바인드 파라미터는 p0, p1... 처럼 안전한 이름 사용
    """
    if df is None or df.empty:
        return

    cols = df.columns.tolist()

    # INSERT columns (quoted identifiers)
    col_sql = ", ".join([f'"{c}"' for c in cols])

    # ✅ safe bind param names
    bind_names = [f"p{i}" for i in range(len(cols))]
    val_sql = ", ".join([f":{bn}" for bn in bind_names])

    # ON CONFLICT key
    key_sql = ", ".join([f'"{c}"' for c in key_cols])

    # UPDATE set = all non-key cols
    non_keys = [c for c in cols if c not in key_cols]
    set_sql = ", ".join([f'"{c}" = EXCLUDED."{c}"' for c in non_keys])

    upsert_sql = f"""
    INSERT INTO {schema}.{table} ({col_sql})
    VALUES ({val_sql})
    ON CONFLICT ({key_sql})
    DO UPDATE SET {set_sql};
    """

    # records: [{p0: v0, p1: v1, ...}, ...]
    records = []
    for row in df.itertuples(index=False, name=None):
        rec = {bn: _to_py(v) for bn, v in zip(bind_names, row)}
        records.append(rec)

    with engine.begin() as conn:
        conn.execute(text(upsert_sql), records)

# -------------------------
# 0) 스키마/테이블 준비
# -------------------------
ensure_schema(engine, SAVE_SCHEMA)

# overall 키 / station 키
KEY_OVERALL = ["prod_day", "shift_type", "pn"]
KEY_STATION = ["prod_day", "shift_type", "station", "pn"]

# 테이블 생성(컬럼은 DF 기준으로 자동)
ensure_table(engine, SAVE_SCHEMA, T_DAY_OVERALL,   df_day_overall.columns.tolist(),   KEY_OVERALL)
ensure_table(engine, SAVE_SCHEMA, T_DAY_STATION,   df_day_station.columns.tolist(),   KEY_STATION)
ensure_table(engine, SAVE_SCHEMA, T_NIGHT_OVERALL, df_night_overall.columns.tolist(), KEY_OVERALL)
ensure_table(engine, SAVE_SCHEMA, T_NIGHT_STATION, df_night_station.columns.tolist(), KEY_STATION)

# -------------------------
# 1) UPSERT 저장
# -------------------------
upsert_df(engine, SAVE_SCHEMA, T_DAY_OVERALL,   df_day_overall,   KEY_OVERALL)
upsert_df(engine, SAVE_SCHEMA, T_DAY_STATION,   df_day_station,   KEY_STATION)
upsert_df(engine, SAVE_SCHEMA, T_NIGHT_OVERALL, df_night_overall, KEY_OVERALL)
upsert_df(engine, SAVE_SCHEMA, T_NIGHT_STATION, df_night_station, KEY_STATION)

print("[INFO] Saved (UPSERT) 4 dataframes into i_daily_report.*")

[INFO] Saved (UPSERT) 4 dataframes into i_daily_report.*
